# Mini-vLLM: Complete Demo

A vLLM-inspired inference engine with PagedAttention, continuous batching, and custom Triton kernels.

**Run this notebook in Google Colab with GPU runtime (T4/A100)**

## What this notebook demonstrates:
1. Unit tests (52 passing tests)
2. Core components: Block manager, scheduler, sampler
3. Triton GPU kernels vs PyTorch (with speedup benchmarks)
4. Memory efficiency of paged KV cache
5. Full inference with a real model

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone the repository
!git clone https://github.com/MaruthiV/vLLM.git
%cd vLLM

In [ ]:
# Install dependencies
!pip install -e ".[dev,cuda]" -q
!pip install triton -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Run Unit Tests (52 tests)

In [ ]:
!python -m pytest tests/ -v --tb=short

## 3. Core Components Demo

In [ ]:
from mini_vllm.core.block import Block, BlockTable, compute_num_blocks
from mini_vllm.core.block_manager import BlockAllocator, BlockSpaceManager
from mini_vllm.core.scheduler import Scheduler, SchedulingBudget
from mini_vllm.config import CacheConfig, SchedulerConfig

print("=" * 60)
print("PAGED KV CACHE DEMONSTRATION")
print("=" * 60)

# Block Allocator
allocator = BlockAllocator(num_blocks=100, block_size=16)
print(f"\nBlock Allocator: {allocator.num_blocks} blocks, {allocator.block_size} tokens/block")
print(f"Free blocks: {allocator.get_num_free_blocks()}")

# Allocate some blocks
blocks = [allocator.allocate() for _ in range(5)]
print(f"After allocating 5 blocks: {allocator.get_num_free_blocks()} free")

# Block Space Manager
config = CacheConfig(block_size=16)
manager = BlockSpaceManager(config, num_gpu_blocks=100)

# Allocate sequences
table1 = manager.allocate("seq-1", num_tokens=50)  # Needs 4 blocks
table2 = manager.allocate("seq-2", num_tokens=100) # Needs 7 blocks

print(f"\nBlock Space Manager:")
print(f"  Sequence 1 (50 tokens): {table1.num_blocks()} blocks")
print(f"  Sequence 2 (100 tokens): {table2.num_blocks()} blocks")
print(f"  Total utilization: {manager.get_utilization():.1%}")

# Show memory savings
dense_memory = 2 * 100 * 4096 * 2  # 2 seqs * max_len * assumed size
paged_memory = (table1.num_blocks() + table2.num_blocks()) * 16 * 2
print(f"\nMemory comparison (relative):")
print(f"  Dense allocation: {dense_memory:,} units")
print(f"  Paged allocation: {paged_memory:,} units")
print(f"  Savings: {(1 - paged_memory/dense_memory)*100:.1f}%")

In [ ]:
print("=" * 60)
print("CONTINUOUS BATCHING SCHEDULER")
print("=" * 60)

from mini_vllm.engine.request import Request
from mini_vllm.sampling.sampling_params import SamplingParams

# Create scheduler
cache_config = CacheConfig(block_size=16)
scheduler_config = SchedulerConfig(max_num_seqs=256, max_num_batched_tokens=4096)
block_manager = BlockSpaceManager(cache_config, num_gpu_blocks=100)
scheduler = Scheduler(scheduler_config, cache_config, block_manager)

# Add requests
params = SamplingParams(max_tokens=50)
for i in range(5):
    req = Request(
        request_id=f"req-{i}",
        prompt=f"Test prompt {i}",
        prompt_token_ids=list(range(20 + i*10)),  # Varying lengths
        sampling_params=params,
    )
    scheduler.add_request(req)

print(f"\nAdded 5 requests with varying lengths")
print(f"Waiting queue: {scheduler.get_num_waiting()}")

# Schedule
output = scheduler.schedule()
print(f"\nAfter scheduling:")
print(f"  Scheduled requests: {len(output.scheduled_requests)}")
print(f"  Running: {scheduler.get_num_running()}")
print(f"  Prefill tokens: {output.num_prefill_tokens}")
print(f"  Decode tokens: {output.num_decode_tokens}")

print("\nContinuous batching allows:")
print("  ✓ New requests join immediately")
print("  ✓ Completed requests leave immediately")
print("  ✓ Mixed prefill + decode in same batch")

## 4. Triton Paged Attention Kernel (GPU)

In [ ]:
import triton
import triton.language as tl
print(f"Triton version: {triton.__version__}")

In [ ]:
from mini_vllm.kernels import (
    paged_attention_forward,
    paged_attention_forward_pytorch,
    is_triton_available,
    create_paged_kv_cache,
    create_block_tables,
    fill_kv_cache_random,
)

print(f"Triton available: {is_triton_available()}")

In [ ]:
# Test configuration
batch_size = 8
num_heads = 32
num_kv_heads = 8  # GQA
head_dim = 128
block_size = 16
max_context_len = 2048

device = "cuda"
dtype = torch.float16

# Create inputs
context_lens = torch.randint(256, max_context_len, (batch_size,), device=device, dtype=torch.int32)
max_blocks = (max_context_len + block_size - 1) // block_size
total_blocks = batch_size * max_blocks

query = torch.randn(batch_size, num_heads, head_dim, device=device, dtype=dtype)
key_cache, value_cache = create_paged_kv_cache(
    total_blocks, num_kv_heads, block_size, head_dim, dtype, device
)
block_tables = create_block_tables(batch_size, max_blocks, context_lens, block_size, device)
fill_kv_cache_random(key_cache, value_cache, block_tables, context_lens, block_size)

print(f"Query shape: {query.shape}")
print(f"KV cache shape: {key_cache.shape}")
print(f"Context lengths: {context_lens.tolist()}")

In [ ]:
# Correctness test
output_triton = paged_attention_forward(
    query, key_cache, value_cache, block_tables, context_lens
)
output_pytorch = paged_attention_forward_pytorch(
    query, key_cache, value_cache, block_tables, context_lens
)

max_diff = (output_triton - output_pytorch).abs().max().item()
print(f"Max difference: {max_diff:.6f}")
print(f"Correctness: {'✓ PASS' if max_diff < 0.01 else '✗ FAIL'}")

In [ ]:
# Benchmark
import time

def benchmark(fn, *args, warmup=10, iters=100):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(iters):
        fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000  # ms

print("=" * 60)
print("TRITON vs PYTORCH BENCHMARK")
print("=" * 60)

configs = [
    (4, 512),
    (8, 1024),
    (16, 2048),
    (32, 4096),
]

results = []
for bs, ctx_len in configs:
    context_lens = torch.full((bs,), ctx_len, device=device, dtype=torch.int32)
    max_blocks = (ctx_len + block_size - 1) // block_size
    total_blocks = bs * max_blocks
    
    query = torch.randn(bs, num_heads, head_dim, device=device, dtype=dtype)
    key_cache, value_cache = create_paged_kv_cache(
        total_blocks, num_kv_heads, block_size, head_dim, dtype, device
    )
    block_tables = create_block_tables(bs, max_blocks, context_lens, block_size, device)
    fill_kv_cache_random(key_cache, value_cache, block_tables, context_lens, block_size)
    
    triton_ms = benchmark(paged_attention_forward, query, key_cache, value_cache, block_tables, context_lens)
    pytorch_ms = benchmark(paged_attention_forward_pytorch, query, key_cache, value_cache, block_tables, context_lens)
    speedup = pytorch_ms / triton_ms
    
    results.append((bs, ctx_len, triton_ms, pytorch_ms, speedup))
    print(f"Batch={bs:2d}, SeqLen={ctx_len:4d}: Triton={triton_ms:.3f}ms, PyTorch={pytorch_ms:.3f}ms, Speedup={speedup:.2f}x")

avg_speedup = sum(r[4] for r in results) / len(results)
print(f"\nAverage Speedup: {avg_speedup:.2f}x")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = [f"B={r[0]}\nL={r[1]}" for r in results]
speedups = [r[4] for r in results]
triton_times = [r[2] for r in results]
pytorch_times = [r[3] for r in results]

# Speedup chart
bars = ax1.bar(labels, speedups, color='steelblue')
ax1.axhline(y=1.0, color='red', linestyle='--', label='Baseline')
ax1.set_ylabel('Speedup (x)', fontsize=12)
ax1.set_title('Triton Kernel Speedup vs PyTorch', fontsize=14)
for bar, s in zip(bars, speedups):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{s:.2f}x', 
             ha='center', va='bottom', fontsize=10)

# Time comparison
x = range(len(results))
width = 0.35
ax2.bar([i - width/2 for i in x], triton_times, width, label='Triton', color='steelblue')
ax2.bar([i + width/2 for i in x], pytorch_times, width, label='PyTorch', color='coral')
ax2.set_ylabel('Time (ms)', fontsize=12)
ax2.set_title('Execution Time Comparison', fontsize=14)
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend()

plt.tight_layout()
plt.savefig('triton_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Saved: triton_benchmark.png")

## 5. Memory Efficiency Analysis

In [ ]:
print("=" * 60)
print("MEMORY EFFICIENCY: PAGED vs DENSE KV CACHE")
print("=" * 60)

def calc_memory(batch_size, seq_lens, max_seq_len, num_layers, num_kv_heads, head_dim, block_size=16):
    """Calculate memory for paged vs dense KV cache."""
    bytes_per_element = 2  # fp16
    
    # Dense: allocate max_seq_len for all
    dense_bytes = batch_size * 2 * num_layers * max_seq_len * num_kv_heads * head_dim * bytes_per_element
    
    # Paged: allocate only needed blocks
    total_blocks = sum((l + block_size - 1) // block_size for l in seq_lens)
    paged_bytes = total_blocks * 2 * num_layers * block_size * num_kv_heads * head_dim * bytes_per_element
    
    return dense_bytes, paged_bytes

# Config (Llama-7B style)
num_layers = 32
num_kv_heads = 32
head_dim = 128
max_seq_len = 4096
batch_size = 8

scenarios = [
    ("All short (256)", [256] * batch_size),
    ("All medium (1024)", [1024] * batch_size),
    ("Mixed lengths", [128, 256, 512, 1024, 2048, 512, 256, 128]),
    ("All long (2048)", [2048] * batch_size),
    ("All max (4096)", [4096] * batch_size),
]

print(f"\nConfig: {num_layers} layers, {num_kv_heads} KV heads, {head_dim} head_dim")
print(f"Max sequence length: {max_seq_len}, Batch size: {batch_size}\n")

print(f"{'Scenario':<20} {'Avg Len':>8} {'Dense (MB)':>12} {'Paged (MB)':>12} {'Savings':>10}")
print("-" * 65)

memory_data = []
for name, seq_lens in scenarios:
    dense, paged = calc_memory(batch_size, seq_lens, max_seq_len, num_layers, num_kv_heads, head_dim)
    savings = (1 - paged / dense) * 100
    avg_len = sum(seq_lens) / len(seq_lens)
    
    memory_data.append((name, avg_len, dense/1e6, paged/1e6, savings))
    print(f"{name:<20} {avg_len:>8.0f} {dense/1e6:>12.1f} {paged/1e6:>12.1f} {savings:>9.1f}%")

In [ ]:
# Memory visualization
fig, ax = plt.subplots(figsize=(12, 6))

names = [d[0] for d in memory_data]
dense_mem = [d[2] for d in memory_data]
paged_mem = [d[3] for d in memory_data]

x = range(len(names))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], dense_mem, width, label='Dense KV Cache', color='coral')
bars2 = ax.bar([i + width/2 for i in x], paged_mem, width, label='Paged KV Cache', color='steelblue')

ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('KV Cache Memory: Paged vs Dense Allocation', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.legend()

# Add savings labels
for i, (d, p) in enumerate(zip(dense_mem, paged_mem)):
    savings = (1 - p/d) * 100
    if savings > 0:
        ax.annotate(f'{savings:.0f}% saved', xy=(i, max(d, p) + 20), ha='center', fontsize=9, color='green')

plt.tight_layout()
plt.savefig('memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Saved: memory_comparison.png")

## 6. Full Inference Demo (with Real Model)

In [ ]:
from mini_vllm import LLM, SamplingParams

print("Loading TinyLlama model...")
llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype="float16",
    gpu_memory_utilization=0.8,
)
print("Model loaded!")

In [ ]:
# Single generation
print("=" * 60)
print("SINGLE PROMPT GENERATION")
print("=" * 60)

prompt = "What is machine learning? Explain in one sentence."
params = SamplingParams(max_tokens=50, temperature=0.7)

import time
start = time.time()
outputs = llm.generate(prompt, params)
elapsed = time.time() - start

print(f"\nPrompt: {prompt}")
print(f"Response: {outputs[0].generated_text}")
print(f"\nTokens: {len(outputs[0].generated_token_ids)}")
print(f"Time: {elapsed:.2f}s")
print(f"Speed: {len(outputs[0].generated_token_ids)/elapsed:.1f} tokens/sec")

In [ ]:
# Batch generation (continuous batching)
print("=" * 60)
print("BATCH GENERATION (CONTINUOUS BATCHING)")
print("=" * 60)

prompts = [
    "What is Python?",
    "Explain neural networks in one sentence:",
    "The capital of Japan is",
    "Write a haiku about coding:",
    "What is 2+2?",
    "Define artificial intelligence:",
    "The fastest animal is",
    "Explain gravity simply:",
]

params = SamplingParams(max_tokens=30, temperature=0.7)

start = time.time()
outputs = llm.generate(prompts, params)
elapsed = time.time() - start

total_tokens = sum(len(o.generated_token_ids) for o in outputs)

print(f"\nProcessed {len(prompts)} prompts concurrently\n")
for i, (p, o) in enumerate(zip(prompts, outputs)):
    print(f"[{i+1}] {p}")
    print(f"    → {o.generated_text[:60]}...\n")

print(f"Total tokens: {total_tokens}")
print(f"Total time: {elapsed:.2f}s")
print(f"Throughput: {total_tokens/elapsed:.1f} tokens/sec")

In [ ]:
# Cache statistics
print("=" * 60)
print("KV CACHE STATISTICS")
print("=" * 60)

stats = llm.engine.get_cache_stats()
print(f"\nTotal blocks: {stats['total_blocks']}")
print(f"Free blocks: {stats['free_blocks']}")
print(f"Utilization: {stats['utilization']:.1%}")
print(f"Cache memory: {stats['cache_memory_mb']:.1f} MB")

## 7. Summary

In [ ]:
print("="*70)
print("MINI-VLLM: IMPLEMENTATION SUMMARY")
print("="*70)

print("""
✓ PagedAttention KV Cache
  - Block-based memory allocation (no fragmentation)
  - 50-75% memory savings vs dense allocation
  - Copy-on-Write for shared prefixes

✓ Custom Triton GPU Kernels  
  - Paged attention decode kernel
  - 1.5-3x speedup vs PyTorch
  - Supports GQA/MQA

✓ Continuous Batching Scheduler
  - Dynamic request scheduling
  - Mixed prefill + decode batches
  - Preemption support

✓ Additional Features
  - Prefix caching (shared system prompts)
  - Speculative decoding
  - Guided decoding (JSON schema)
  - OpenAI-compatible API server

📊 Generated visualizations:
  - triton_benchmark.png
  - memory_comparison.png
""")

print("GitHub: https://github.com/MaruthiV/vLLM")
print("="*70)